# 06 — Final evaluation, risk-policy review and artifact export

**Purpose.** Evaluate the staged candidates on the **untouched test split** (first and only use of test labels for
model comparison), review the combined system with the backend's risk policy, and export the deployment bundle that the
API loads (manifest, hashes, golden parity fixtures, model card, registry entry).

Reported per model: precision, recall, F1 (macro and weighted), ROC-AUC/PR-AUC where defined, false-alarm rate,
confusion matrix, training time, inference latency and model size — all measured here, nothing is pre-filled.

In [ ]:
# --- Project bootstrap: make backend/app/ml and notebooks/nb_utils importable ---------------------
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not ((_root / "backend" / "app" / "ml").is_dir() and (_root / "notebooks").is_dir()):
    if _root.parent == _root:
        raise RuntimeError("Start Jupyter inside the CyberSentinel repository (for example from notebooks/).")
    _root = _root.parent
for _path in (str(_root / "backend"), str(_root / "notebooks")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

from nb_utils import load_config

CFG = load_config()  # all settings come from CS_* environment variables (see nb_utils/config.py)
CFG.describe()

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

from app.ml.artifacts import ExpectedBundle, load_bundle, verify_golden
from app.ml.artifacts.export import export_bundle
from app.ml.artifacts.manifest import DatasetSection, FeatureReference, InputFeature
from app.ml.constants import BENIGN_LABEL, MALICIOUS_LABEL, AnalysisMode, ExplanationMethod, ModelType
from app.ml.datasets import get_schema_for_family
from app.ml.evaluation import metrics as M
from app.ml.inference import InferenceEngine
from app.ml.risk import AnomalyReference, compute_risk, load_policy
from app.ml.risk.policy import Weights
from nb_utils import io, plotting, splits
from nb_utils.experiments import record_experiment, save_json

warnings.filterwarnings("ignore")
SCHEMA = get_schema_for_family(CFG.family)
SUP, ANOM, SEMI = (CFG.staging_dir / d for d in ("supervised", "anomaly", "semi_supervised"))

In [ ]:
splits_ = io.load_splits(CFG.processed_dir)
prep_report = io.load_json(CFG.processed_dir / "preprocessing_report.json")
preprocessor = io.load_object(CFG.processed_dir / "preprocessor.joblib")
selection = io.load_json(SUP / "selection.json")
anomaly_meta = io.load_json(ANOM / "anomaly_meta.json")
iso = io.load_object(ANOM / "isolation_forest.joblib")
FEATURES = [f["name"] for f in prep_report["input_features"]]
val, test = splits_["validation"], splits_["test"]
X_val, X_test = preprocessor.transform(val[FEATURES]), preprocessor.transform(test[FEATURES])
MC_CLASSES = selection["multiclass"]["classes"]
calibration = selection["multiclass"].get("calibration")
# The deployed multi-class classifier is the calibrated wrapper when calibration was performed in notebook 03.
DEPLOYED_KEY = "multiclass__calibrated" if calibration else selection["multiclass"]["key"]
print("test rows:", len(test), "| staged supervised models:", sorted(p.stem for p in SUP.glob("*__*.joblib")))
print("deployed multi-class classifier:", DEPLOYED_KEY)

## 1. Test-set evaluation of all staged classifiers

In [ ]:
def evaluate(model, X, labels_true, classes, threshold=None):
    labels_true = np.asarray(labels_true, dtype=object)
    known = np.isin(labels_true, classes)
    X, labels_true = X[known], labels_true[known]
    proba = model.predict_proba(X)
    index = (proba[:, 1] >= threshold).astype(int) if threshold is not None else proba.argmax(axis=1)
    labels_pred = np.asarray(classes, dtype=object)[index]
    report = M.classification_report_dict(labels_true, labels_pred, classes)
    is_attack = (labels_true != BENIGN_LABEL).astype(int)
    attack_score = 1.0 - proba[:, classes.index(BENIGN_LABEL)]
    out = {"rows_evaluated": int(known.sum()), "rows_with_unseen_labels": int((~known).sum()),
           "macro_precision": report["macro"]["precision"], "macro_recall": report["macro"]["recall"],
           "macro_f1": report["macro"]["f1"], "weighted_f1": report["weighted"]["f1"], "accuracy": report["accuracy"],
           "false_alarm_rate": M.false_alarm_rate(labels_true, labels_pred, BENIGN_LABEL),
           "attack_roc_auc": M.roc_auc_safe(is_attack, attack_score), "attack_pr_auc": M.pr_auc_safe(is_attack, attack_score)}
    if len(classes) > 2:
        out["ovr_roc_auc_macro"] = M.multiclass_roc_auc_ovr(labels_true, proba, classes)["macro"]
    latency = M.measure_latency(model.predict_proba, X[: min(len(X), 10_000)])
    out.update(ms_per_record=latency["ms_per_record"], records_per_second=latency["records_per_second"])
    return out, report, M.confusion(labels_true, labels_pred, classes)


test_labels = {"multiclass": test["label_category"].to_numpy(),
               "binary": np.where(test["label_binary"].to_numpy() == 1, MALICIOUS_LABEL, BENIGN_LABEL)}
rows, test_reports, test_confusions, test_metrics = [], {}, {}, {}
for path in sorted(SUP.glob("*__*.joblib")):
    task, name = path.stem.split("__", 1)
    if name == "calibrated":
        name = f"{selection['multiclass']['model']}_calibrated"
    model = io.load_object(path)
    classes = selection[task]["classes"]
    threshold = selection["binary"]["decision_threshold"] if (task == "binary" and name == selection["binary"]["model"]) else None
    metrics_test, report, cm = evaluate(model, X_test, test_labels[task], classes, threshold)
    test_reports[path.stem], test_confusions[path.stem], test_metrics[path.stem] = report, cm, metrics_test
    rows.append({"task": task, "model": name, "split": "test", "selected_on_validation": name == selection[task]["model"],
                 "deployed": path.stem in (DEPLOYED_KEY, selection["binary"]["key"]),
                 "train_seconds": selection["train_seconds"].get(path.stem), "size_mb": round(path.stat().st_size / 1e6, 3), **metrics_test})
test_results = pd.DataFrame(rows)
test_results.sort_values(["task", "macro_f1"], ascending=[True, False])

In [ ]:
for task, key in (("multiclass", DEPLOYED_KEY), ("binary", selection["binary"]["key"])):
    cm = test_confusions[key]
    plotting.confusion_heatmap(cm["matrix"], cm["labels"], f"{key} — test (row-normalized)", CFG.figures_dir / f"06_confusion_{task}.png")
pd.DataFrame(test_reports[DEPLOYED_KEY]["per_class"]).T

In [ ]:
record_experiment(
    experiment_id=f"{CFG.family}-supervised-test", experiment_type="supervised_comparison",
    title="Supervised candidates (test split)", family=CFG.family, rows=test_results,
    columns={"macro_f1": "macro-averaged F1 on the test split", "false_alarm_rate": "share of benign test flows predicted as attacks",
             "selected_on_validation": "model chosen in notebook 03 using validation data only"},
    config={"seed": CFG.seed, "binary_decision_threshold": selection["binary"]["decision_threshold"]},
    summary={"selected": {t: selection[t]["model"] for t in ("multiclass", "binary")}},
    notes=["Model selection used the validation split; these test results were computed afterwards."],
    experiments_dir=CFG.experiments_dir, reports_dir=CFG.reports_dir)
print("test comparison recorded")

## 2. Anomaly detector on the test split (threshold fixed on validation)

In [ ]:
anomaly_test = M.anomaly_detection_metrics(test["label_binary"].to_numpy(), -iso.score_samples(X_test),
                                           anomaly_meta["threshold"], test["label_category"].to_numpy())
print({k: anomaly_test[k] for k in ("roc_auc", "pr_auc", "flag_rate")}, anomaly_test["at_threshold"])
pd.DataFrame(anomaly_test["flag_rate_by_category"]).T

## 3. Combined system with the backend risk policy
Applies `backend/config/risk_policy.json` (the same code the API runs) to the selected classifier + anomaly detector. The weight sweep uses the **validation** split; a candidate policy is written to `reports/` for your review and is **not** applied automatically.

In [ ]:
policy = load_policy(CFG.project_root / "backend" / "config" / "risk_policy.json")
mc_model = io.load_object(SUP / f"{DEPLOYED_KEY}.joblib")  # the classifier that will be deployed
reference = AnomalyReference(**anomaly_meta["reference_points"])


def system_view(X, labels_true, active_policy):
    proba = mc_model.predict_proba(X)
    scores = -iso.score_samples(X)
    risk = compute_risk(active_policy, size=len(X), class_probabilities=proba, classes=MC_CLASSES, benign_class=BENIGN_LABEL,
                        anomaly_scores=scores, anomaly_reference=reference, is_anomaly=scores >= anomaly_meta["threshold"],
                        needs_review=proba.max(axis=1) < selection["multiclass"]["min_confidence"])
    is_attack = np.asarray(labels_true) != BENIGN_LABEL
    alerts = risk.is_alert
    summary = {"alert_rate": float(alerts.mean()),
               "attack_recall_at_alert": float(alerts[is_attack].mean()) if is_attack.any() else None,
               "alert_precision": float(is_attack[alerts].mean()) if alerts.any() else None,
               "false_alerts_per_10k_benign": float(10_000 * alerts[~is_attack].mean()) if (~is_attack).any() else None}
    table = pd.crosstab(pd.Series(labels_true, name="true category"), pd.Series(risk.band, name="risk band"))
    return summary, table


val_summary, val_table = system_view(X_val, val["label_category"].to_numpy(), policy)
test_summary, test_table = system_view(X_test, test["label_category"].to_numpy(), policy)
print("policy", policy.policy_version, "| validation:", val_summary, "\n test:", test_summary)
test_table

In [ ]:
sweep = []
for w in (0.5, 0.6, 0.7, 0.8, 0.9):
    variant = policy.model_copy(update={"weights": Weights(classifier=w, anomaly=round(1 - w, 10))})
    summary, _ = system_view(X_val, val["label_category"].to_numpy(), variant)
    p, r = summary["alert_precision"] or 0.0, summary["attack_recall_at_alert"] or 0.0
    sweep.append({"classifier_weight": w, **summary, "alert_f1": 2 * p * r / (p + r) if p + r else 0.0})
sweep = pd.DataFrame(sweep)
best_w = float(sweep.sort_values("alert_f1", ascending=False).iloc[0]["classifier_weight"])
candidate = policy.model_copy(update={"weights": Weights(classifier=best_w, anomaly=round(1 - best_w, 10))})
save_json(candidate.model_dump(mode="json"), CFG.reports_dir / "risk_policy.candidate.json")
record_experiment(
    experiment_id=f"{CFG.family}-risk-policy-evaluation", experiment_type="risk_policy_evaluation",
    title="Risk-policy weight sweep (validation split)", family=CFG.family, rows=sweep,
    columns={"classifier_weight": "weight of the classifier component (anomaly weight = 1 - w)",
             "attack_recall_at_alert": "share of attack flows that become alerts", "alert_precision": "share of alerts that are attacks"},
    config={"base_policy_version": policy.policy_version},
    summary={"active_policy_validation": val_summary, "active_policy_test": test_summary, "candidate_classifier_weight": best_w},
    notes=["The candidate policy is written to reports/ and is not applied automatically; review before copying it to backend/config."],
    experiments_dir=CFG.experiments_dir, reports_dir=CFG.reports_dir)
sweep

## 4. Export the deployment bundle
Selected multi-class classifier + Isolation Forest + the fitted preprocessor. The exporter validates the manifest, records SHA-256 hashes and runtime versions, writes golden parity fixtures and a model card, and adds the bundle to `model_artifacts/registry.json`.

In [ ]:
algorithm = selection["multiclass"]["model"]
classifier = mc_model
explanation_method = (ExplanationMethod.BASELINE_DEVIATION if calibration else
                      {"xgboost": ExplanationMethod.XGBOOST_TREE_SHAP, "random_forest": ExplanationMethod.TREE_PATH_ATTRIBUTION}
                      .get(algorithm, ExplanationMethod.BASELINE_DEVIATION))
MODEL_NAME = f"{CFG.family.replace('_', '-')}-{algorithm.replace('_', '-')}-multiclass"
split = prep_report["split"]
dataset = DatasetSection(
    family=CFG.family, schema_version=SCHEMA.schema_version, source=SCHEMA.source, split_strategy=split["strategy"],
    random_seed=CFG.seed, train_rows=split["sizes"]["train"], validation_rows=split["sizes"]["validation"],
    test_rows=split["sizes"]["test"], label_mapping_version=SCHEMA.label_mapping_version,
    class_policy=(f"classes too rare to stratify were kept in training only: {split['rare_classes_train_only']}"
                  if split["rare_classes_train_only"] else None))
metrics_payload = {
    "evaluation_note": ("validation metrics (of the uncalibrated base model) guided selection; test metrics of the deployed "
                        "classifier were computed once afterwards"),
    "validation": selection["validation_reports"][selection["multiclass"]["key"]],
    "test": test_metrics[DEPLOYED_KEY],
    "test_per_class": test_reports[DEPLOYED_KEY]["per_class"],
    "test_confusion_matrix": test_confusions[DEPLOYED_KEY],
    "anomaly_detector": {"validation": anomaly_meta["validation"], "test": anomaly_test},
    "risk_policy": {"policy_version": policy.policy_version, "validation": val_summary, "test": test_summary},
}
limitations = [
    f"Trained and evaluated on {SCHEMA.display_name} only; performance on other networks or datasets is untested.",
    "Benchmark datasets contain known label errors and lab-specific artefacts that can inflate reported metrics.",
    "Classifier scores are " + ("calibrated (see calibration section)." if calibration else "not calibrated probabilities."),
    "Anomaly scores measure unusualness relative to training traffic; anomalies are not necessarily attacks.",
    f"Shortcut-feature candidates ({', '.join(SCHEMA.shortcut_feature_candidates)}) were "
    + ("removed." if CFG.drop_shortcut_features else f"kept; see the notebook 03 ablation: {selection.get('shortcut_ablation')}"),
]
golden = splits.stratified_sample(test, "label_category", 200, CFG.seed)[FEATURES]
result = export_bundle(
    model_dir=CFG.model_dir, model_name=MODEL_NAME, version=CFG.model_version, model_type=ModelType.ENSEMBLE,
    description=f"{algorithm.replace('_', ' ').title()} multi-class classifier with an Isolation Forest anomaly detector for {SCHEMA.display_name} flows.",
    created_by="notebooks/06_model_evaluation.ipynb", dataset=dataset, preprocessor=preprocessor,
    preprocessing_version=prep_report["preprocessing_version"],
    preprocessing_description="median imputation, signed log1p, standardization; one-hot encoding for categorical features",
    preprocessing_steps=prep_report["preprocessing_steps"],
    input_features=[InputFeature(**f) for f in prep_report["input_features"]],
    benign_reference={k: FeatureReference(**v) for k, v in prep_report["benign_reference"].items()},
    classifier=classifier,
    classifier_meta={"task": "multiclass", "classes": MC_CLASSES, "benign_class": BENIGN_LABEL,
                     "probability_calibrated": bool(calibration), "calibration": calibration,
                     "min_confidence": selection["multiclass"]["min_confidence"], "explanation_method": explanation_method,
                     "global_feature_importance": selection["multiclass"]["global_feature_importance"]},
    anomaly_detector=iso,
    anomaly_meta={k: anomaly_meta[k] for k in ("score_definition", "threshold", "threshold_method", "reference_points", "training_population")},
    metrics=metrics_payload, limitations=limitations, golden_frame=golden,
    overwrite=os.environ.get("CS_OVERWRITE_BUNDLE", "0") == "1",
)
print("bundle:", result.bundle_dir.relative_to(CFG.model_dir), "| manifest sha256:", result.manifest_sha256)

## 5. Verify the exported bundle exactly as the API will load it

In [ ]:
bundle_path = f"bundles/{MODEL_NAME}/{CFG.model_version}"
loaded = load_bundle(CFG.model_dir, bundle_path,
                     ExpectedBundle(MODEL_NAME, CFG.model_version, result.manifest_sha256, SCHEMA.schema_version, prep_report["preprocessing_version"]),
                     max_artifact_bytes=8 * 1024**3)
golden_report = verify_golden(loaded)
assert golden_report["passed"], golden_report
sample = test[FEATURES].head(500)
engine_out = InferenceEngine(loaded).predict(sample, AnalysisMode.COMBINED)
direct = classifier.predict_proba(preprocessor.transform(sample))
assert np.allclose(engine_out.class_probabilities, direct, atol=1e-12), "engine output differs from notebook predictions"
print("golden parity:", golden_report)
print(f"""
Next steps (run from backend/ with the API's virtual environment):
  python -m app.cli models sync                       # register bundles listed in model_artifacts/registry.json as draft
  python -m app.cli models approve {MODEL_NAME} {CFG.model_version}
  python -m app.cli models verify {MODEL_NAME} {CFG.model_version}
""")

## 6. Optional additional bundles
* `CS_EXPORT_BINARY=1` — the selected binary classifier (with its validation-chosen decision threshold) + Isolation Forest.
* A Label Spreading model staged by notebook 05 (`CS_EXPORT_SEMI=1`) is exported as a `semi_supervised` bundle.

In [ ]:
common = dict(model_dir=CFG.model_dir, version=CFG.model_version, created_by="notebooks/06_model_evaluation.ipynb", dataset=dataset,
              preprocessor=preprocessor, preprocessing_version=prep_report["preprocessing_version"],
              preprocessing_description="median imputation, signed log1p, standardization; one-hot encoding for categorical features",
              preprocessing_steps=prep_report["preprocessing_steps"], input_features=[InputFeature(**f) for f in prep_report["input_features"]],
              benign_reference={k: FeatureReference(**v) for k, v in prep_report["benign_reference"].items()}, golden_frame=golden,
              overwrite=os.environ.get("CS_OVERWRITE_BUNDLE", "0") == "1")
exported = [MODEL_NAME]
if os.environ.get("CS_EXPORT_BINARY", "0") == "1":
    binary_algorithm = selection["binary"]["model"]
    name = f"{CFG.family.replace('_', '-')}-{binary_algorithm.replace('_', '-')}-binary"
    export_bundle(**common, model_name=name, model_type=ModelType.ENSEMBLE,
                  description=f"Binary benign/malicious {binary_algorithm.replace('_', ' ')} classifier with an Isolation Forest.",
                  classifier=io.load_object(SUP / f"{selection['binary']['key']}.joblib"),
                  classifier_meta={"task": "binary", "classes": selection["binary"]["classes"], "benign_class": BENIGN_LABEL,
                                   "decision_threshold": selection["binary"]["decision_threshold"],
                                   "threshold_method": selection["binary"]["threshold_method"], "min_confidence": 0.0,
                                   "explanation_method": {"xgboost": ExplanationMethod.XGBOOST_TREE_SHAP, "random_forest": ExplanationMethod.TREE_PATH_ATTRIBUTION}.get(binary_algorithm, ExplanationMethod.BASELINE_DEVIATION)},
                  anomaly_detector=iso,
                  anomaly_meta={k: anomaly_meta[k] for k in ("score_definition", "threshold", "threshold_method", "reference_points", "training_population")},
                  metrics={"test": test_metrics[selection["binary"]["key"]], "decision_threshold": selection["binary"]["decision_threshold"]},
                  limitations=limitations)
    exported.append(name)
if (SEMI / "label_spreading.joblib").is_file():
    semi_meta = io.load_json(SEMI / "semi_meta.json")
    name = f"{CFG.family.replace('_', '-')}-label-spreading-multiclass"
    export_bundle(**common, model_name=name, model_type=ModelType.SEMI_SUPERVISED,
                  description=f"Label Spreading (knn graph) trained with {semi_meta['labeled_samples']} labels from a {semi_meta['pool_rows']}-row pool.",
                  classifier=io.load_object(SEMI / "label_spreading.joblib"),
                  classifier_meta={"task": "multiclass", "classes": semi_meta["classes"], "benign_class": BENIGN_LABEL, "min_confidence": 0.5,
                                   "explanation_method": ExplanationMethod.BASELINE_DEVIATION},
                  metrics={"test_sample": semi_meta["test_sample_metrics"], "label_budget": semi_meta["label_budget"]},
                  limitations=[*limitations, "The model stores its training pool; latency and memory grow with the pool size."])
    exported.append(name)
exported